# 00 — Reset demo schema

Drops and recreates the Unity Catalog schema before seeding demo tables (`short_term_*`, `volume_forecast_*`).

Assumes the **catalog** already exists in the workspace (typical for shared demos). `CREATE CATALOG` runs only when the catalog is missing.

**Default:** `energy_utilities.energy_trading2` — override via job parameters / widgets `catalog` and `schema`.

In [ ]:
# Databricks notebook source
import os

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"

print(f"Resetting: {CATALOG}.{SCHEMA}")

# Catalog is provisioned by platform admins in most workspaces — create only when missing.
_catalog_names = {c.name for c in spark.catalog.listCatalogs()}
if CATALOG not in _catalog_names:
    print(f"Catalog `{CATALOG}` not found — creating.")
    spark.sql(f"CREATE CATALOG `{CATALOG}`")
else:
    print(f"Catalog `{CATALOG}` already exists — skipping CREATE CATALOG.")

spark.sql(f"DROP SCHEMA IF EXISTS `{CATALOG}`.`{SCHEMA}` CASCADE")
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}` "
    f"COMMENT 'Energy trading demo data (short_term_*, volume_forecast_*; reset by workflow job)'"
)

print(f"Schema ready: {CATALOG}.{SCHEMA}")